In [3]:
"""
╔══════════════════════════════════════════════════════════════════╗
║         MODALITY-FLOW — ML PIPELINE                             ║
║         Sprint 4 — Modèle Prédictif                             ║
╠══════════════════════════════════════════════════════════════════╣
║  Modèles:                                                       ║
║  1. Bisiklet doluluk tahmini (RandomForest Regressor)           ║
║  2. En az karbonlu rota tahmini                                 ║
║  3. Kişisel CO₂ tasarrufu hesabı                               ║
╚══════════════════════════════════════════════════════════════════╝

LANCER:
    pip3 install scikit-learn
    python3 ml_pipeline.py
"""

import sys
import logging
import warnings
import duckdb
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════

VELO_DIR    = Path.home() / "Desktop" / "Velo"
ETL_DIR     = VELO_DIR / "ETL"
DUCKDB_PATH = ETL_DIR / "gold" / "modality_flow.duckdb"
ML_DIR      = VELO_DIR / "ML"
MODELS_DIR  = ML_DIR / "models"
REPORTS_DIR = ML_DIR / "reports"

CO2_FACTORS = {
    "velo":    0,
    "tram":    4,
    "bus":     68,
    "voiture": 120,
    "marche":  0,
}

# ══════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════

def setup_logging():
    ML_DIR.mkdir(parents=True, exist_ok=True)
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    fmt = "%(asctime)s [%(levelname)s] %(message)s"
    logging.basicConfig(
        level=logging.INFO, format=fmt, datefmt="%Y-%m-%d %H:%M:%S",
        handlers=[
            logging.FileHandler(ML_DIR / "ml_pipeline.log", encoding="utf-8"),
            logging.StreamHandler(sys.stdout)
        ]
    )
    return logging.getLogger("modality_ml")

log = setup_logging()


# ══════════════════════════════════════════════════════════════════
# 1. VERI HAZIRLAMA
# ══════════════════════════════════════════════════════════════════

def load_features() -> pd.DataFrame:
    """
    DuckDB'den ML feature'larını yükle.
    fact_velomagg_historique + dim_meteo + dim_qualite_air JOIN.
    """
    log.info("=" * 65)
    log.info("Veri yükleniyor...")
    log.info("=" * 65)

    con = duckdb.connect(str(DUCKDB_PATH))

    df = con.execute("""
        SELECT
            h.station_id,
            h.timestamp,
            h.bisiklet_sayisi,
            EXTRACT(HOUR  FROM h.timestamp) AS heure,
            EXTRACT(DOW   FROM h.timestamp) AS jour_semaine,
            EXTRACT(MONTH FROM h.timestamp) AS mois,
            EXTRACT(DAY   FROM h.timestamp) AS jour_mois,
            -- Heure de pointe
            CASE
                WHEN EXTRACT(HOUR FROM h.timestamp) BETWEEN 7 AND 9  THEN 1
                WHEN EXTRACT(HOUR FROM h.timestamp) BETWEEN 17 AND 19 THEN 1
                ELSE 0
            END AS heure_pointe,
            -- Weekend
            CASE
                WHEN EXTRACT(DOW FROM h.timestamp) IN (0, 6) THEN 1
                ELSE 0
            END AS weekend,
            -- Météo (LEFT JOIN — NaN olabilir)
            COALESCE(m.temperature_max, 15.0)   AS temperature_max,
            COALESCE(m.temperature_min, 8.0)    AS temperature_min,
            COALESCE(m.precipitation_sum, 0.0)  AS precipitation_sum,
            COALESCE(m.wind_speed_max, 10.0)    AS wind_speed_max,
            COALESCE(m.weather_code, 0)         AS weather_code,
            -- AQI (LEFT JOIN — NaN olabilir)
            COALESCE(q.indice_qualite, 3)       AS indice_qualite,
            COALESCE(q.no2, 10)                 AS no2,
            COALESCE(q.o3, 50)                  AS o3,
            COALESCE(q.pm10, 15)                AS pm10
        FROM fact_velomagg_historique h
        LEFT JOIN dim_meteo m
            ON CAST(h.timestamp AS DATE) = m.date
        LEFT JOIN dim_qualite_air q
            ON CAST(h.timestamp AS DATE) = q.date
        WHERE h.bisiklet_sayisi IS NOT NULL
          AND h.bisiklet_sayisi >= 0
    """).fetchdf()

    con.close()

    log.info(f"{len(df)} satır yüklendi")
    log.info(f"   Kolonlar: {df.columns.tolist()}")
    log.info(f"   İstasyon sayısı: {df['station_id'].nunique()}")
    log.info(f"   Tarih aralığı: {df['timestamp'].min()} → {df['timestamp'].max()}")
    log.info(f"   Bisiklet min/max: {df['bisiklet_sayisi'].min()} / {df['bisiklet_sayisi'].max()}")

    return df


# ══════════════════════════════════════════════════════════════════
# 2. MODEL 1 — BİSİKLET DOLULUK TAHMİNİ
# ══════════════════════════════════════════════════════════════════

def train_availability_model(df: pd.DataFrame):
    """
    RandomForest Regressor ile bisiklet doluluk tahmini.
    
    TARGET: bisiklet_sayisi (kaç bisiklet mevcut?)
    FEATURES:
        - heure (0-23)
        - jour_semaine (0=Pazartesi, 6=Pazar)
        - mois (1-12)
        - jour_mois (1-31)
        - heure_pointe (0/1)
        - weekend (0/1)
        - temperature_max
        - precipitation_sum
        - wind_speed_max
        - indice_qualite (AQI)
        - station_id (encoded)
    """
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error, r2_score

    log.info("=" * 65)
    log.info("🤖 MODEL 1 — Bisiklet Doluluk Tahmini")
    log.info("=" * 65)

    # Feature'lar
    FEATURES = [
        "heure", "jour_semaine", "mois", "jour_mois",
        "heure_pointe", "weekend",
        "temperature_max", "precipitation_sum", "wind_speed_max",
        "indice_qualite", "no2", "o3", "pm10",
        "station_encoded"
    ]
    TARGET = "bisiklet_sayisi"

    # Station ID encoding
    le = LabelEncoder()
    df["station_encoded"] = le.fit_transform(df["station_id"])

    # Train/test split (80/20)
    X = df[FEATURES]
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    log.info(f"   Train: {len(X_train)} satır | Test: {len(X_test)} satır")

    # Model eğitimi
    log.info("   RandomForest eğitiliyor...")
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # Değerlendirme
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    log.info(f"   MAE  (Mean Absolute Error): {mae:.2f} bisiklet")
    log.info(f"   R²   (Coefficient): {r2:.3f}")
    log.info(f"   Yorum: Ortalama {mae:.1f} bisiklet hata payıyla tahmin yapıyor")

    # Feature importance
    importances = pd.DataFrame({
        "feature": FEATURES,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    log.info(f"\n   En önemli feature'lar:\n{importances.head(7).to_string(index=False)}")

    # Modeli kaydet
    model_path = MODELS_DIR / "availability_model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump({"model": model, "encoder": le, "features": FEATURES}, f)
    log.info(f"Model kaydedildi: {model_path}")

    # Örnek tahmin
    log.info("")
    log.info("   Örnek tahminler:")
    test_cases = [
        {"station": "001", "heure": 8,  "jour": 1, "mois": 5, "pointe": 1, "weekend": 0, "desc": "Pazartesi sabah 08:00"},
        {"station": "001", "heure": 12, "jour": 3, "mois": 5, "pointe": 0, "weekend": 0, "desc": "Çarşamba öğle 12:00"},
        {"station": "001", "heure": 18, "jour": 5, "mois": 5, "pointe": 1, "weekend": 0, "desc": "Cuma akşam 18:00"},
        {"station": "001", "heure": 14, "jour": 6, "mois": 7, "pointe": 0, "weekend": 1, "desc": "Cumartesi öğleden sonra"},
    ]

    for tc in test_cases:
        station_enc = le.transform([tc["station"]])[0] if tc["station"] in le.classes_ else 0
        X_pred = pd.DataFrame([{
            "heure":            tc["heure"],
            "jour_semaine":     tc["jour"],
            "mois":             tc["mois"],
            "jour_mois":        15,
            "heure_pointe":     tc["pointe"],
            "weekend":          tc["weekend"],
            "temperature_max":  22.0,
            "precipitation_sum":0.0,
            "wind_speed_max":   15.0,
            "indice_qualite":   2,
            "no2":              10,
            "o3":               50,
            "pm10":             15,
            "station_encoded":  station_enc,
        }])
        pred = model.predict(X_pred)[0]
        log.info(f"   {tc['desc']}: ~{pred:.0f} bisiklet mevcut")

    return model, le


# ══════════════════════════════════════════════════════════════════
# 3. MODEL 2 — EN AZ KARBONLU ROTA TAHMİNİ
# ══════════════════════════════════════════════════════════════════

def compute_optimal_route(
    lat_a: float, lon_a: float,
    lat_b: float, lon_b: float,
    heure: int = 8,
    jour_semaine: int = 1,
    precipitation: float = 0.0,
    bikes_available: int = 5
) -> dict:
    """
    A noktasından B noktasına en az karbonlu rotayı hesapla.
    
    Haversine formülü ile mesafe hesabı.
    CO₂ faktörleri ADEME 2024.
    """
    # Haversine mesafe (km)
    R = 6371
    dlat = np.radians(lat_b - lat_a)
    dlon = np.radians(lon_b - lon_a)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat_a)) * np.cos(np.radians(lat_b)) * np.sin(dlon/2)**2
    distance_km = R * 2 * np.arcsin(np.sqrt(a))

    # Her mod için CO₂ hesabı
    routes = []
    for mode, co2_per_km in CO2_FACTORS.items():
        # Hız (km/h)
        speeds = {"velo": 15, "tram": 25, "bus": 20, "voiture": 30, "marche": 5}
        speed = speeds.get(mode, 20)
        duration_min = round(distance_km / speed * 60, 1)
        co2_total_g = round(distance_km * co2_per_km, 1)
        co2_saved_vs_car = round(distance_km * CO2_FACTORS["voiture"] - co2_total_g, 1)

        # Uygunluk skoru (daha düşük = daha iyi)
        score = co2_total_g

        # Yağmurda bisiklet cezası
        if mode == "velo" and precipitation > 5:
            score += 50  # Yağmurda bisiklet önerilmez
        
        # Bisiklet yoksa ceza
        if mode == "velo" and bikes_available == 0:
            score += 100  # İstasyon boşsa önerilmez

        # 5 km'den fazla yürüme önerilmez
        if mode == "marche" and distance_km > 2:
            score += 200

        routes.append({
            "mode":              mode,
            "distance_km":       round(distance_km, 2),
            "duration_min":      duration_min,
            "co2_g":             co2_total_g,
            "co2_saved_vs_car":  co2_saved_vs_car,
            "score":             score,
            "recommande":        False
        })

    # En iyi rotayı seç
    routes.sort(key=lambda x: x["score"])
    routes[0]["recommande"] = True

    return {
        "distance_km":    round(distance_km, 2),
        "routes":         routes,
        "best_mode":      routes[0]["mode"],
        "co2_economise":  routes[0]["co2_saved_vs_car"],
    }


def demo_route_optimization():
    """Rota optimizasyonu demo."""
    log.info("=" * 65)
    log.info("🗺️  MODEL 2 — En Az Karbonlu Rota Tahmini")
    log.info("=" * 65)

    # Örnek rotalar — Montpellier
    test_routes = [
        {
            "desc": "Gare Saint-Roch → Place de la Comédie",
            "lat_a": 43.6047, "lon_a": 3.8767,
            "lat_b": 43.6089, "lon_b": 3.8795,
            "heure": 8, "precipitation": 0.0, "bikes": 8
        },
        {
            "desc": "Antigone → Faculté de Médecine",
            "lat_a": 43.6089, "lon_a": 3.8920,
            "lat_b": 43.6150, "lon_b": 3.8680,
            "heure": 9, "precipitation": 0.0, "bikes": 3
        },
        {
            "desc": "Centre → Odysseum (temps de pluie)",
            "lat_a": 43.6089, "lon_a": 3.8795,
            "lat_b": 43.6200, "lon_b": 3.9200,
            "heure": 18, "precipitation": 10.0, "bikes": 5
        },
    ]

    for tr in test_routes:
        result = compute_optimal_route(
            tr["lat_a"], tr["lon_a"],
            tr["lat_b"], tr["lon_b"],
            heure=tr["heure"],
            precipitation=tr["precipitation"],
            bikes_available=tr["bikes"]
        )

        log.info(f"\n   📍 {tr['desc']}")
        log.info(f"      Distance: {result['distance_km']} km")
        log.info(f"      Meilleur mode: {result['best_mode'].upper()} ⭐")
        log.info(f"      CO₂ économisé vs voiture: {result['co2_economise']} g")
        log.info(f"      Comparaison:")

        for r in result["routes"]:
            star = "RECOMMANDÉ" if r["recommande"] else ""
            log.info(f"        {r['mode']:<10} {r['distance_km']} km | {r['duration_min']} min | {r['co2_g']} g CO₂{star}")


# ══════════════════════════════════════════════════════════════════
# 4. MODEL 3 — KİŞİSEL CO₂ TASARRUFU
# ══════════════════════════════════════════════════════════════════

def compute_personal_co2_savings(
    trips: list,
    mode_utilise: str = "velo",
    mode_reference: str = "voiture"
) -> dict:
    """
    Kullanıcının kişisel CO₂ tasarrufunu hesapla.
    
    trips: [{"distance_km": 3.5, "date": "2026-05-12"}, ...]
    """
    total_distance = sum(t["distance_km"] for t in trips)
    co2_reference  = total_distance * CO2_FACTORS[mode_reference]
    co2_utilise    = total_distance * CO2_FACTORS[mode_utilise]
    co2_economise  = co2_reference - co2_utilise

    return {
        "nb_trajets":       len(trips),
        "distance_totale":  round(total_distance, 2),
        "co2_reference_g":  round(co2_reference, 1),
        "co2_utilise_g":    round(co2_utilise, 1),
        "co2_economise_g":  round(co2_economise, 1),
        "co2_economise_kg": round(co2_economise / 1000, 3),
        "arbres_equivalent":round(co2_economise / 22000, 2),  # 1 arbre = 22 kg CO₂/an
        "mode_utilise":     mode_utilise,
        "mode_reference":   mode_reference,
    }


def demo_co2_savings():
    """CO₂ tasarrufu demo."""
    log.info("=" * 65)
    log.info("🌱 MODEL 3 — Kişisel CO₂ Tasarrufu")
    log.info("=" * 65)

    # Örnek kullanıcı — 1 ay bisiklet kullanımı
    trips_1mois = [{"distance_km": np.random.uniform(1, 8), "date": f"2026-05-{i:02d}"} for i in range(1, 31)]

    result = compute_personal_co2_savings(trips_1mois, mode_utilise="velo", mode_reference="voiture")

    log.info(f"   Kullanıcı profili (1 ay):")
    log.info(f"   Trajet sayısı:        {result['nb_trajets']}")
    log.info(f"   Toplam mesafe:        {result['distance_totale']} km")
    log.info(f"   Araba ile olsaydı:    {result['co2_reference_g']} g CO₂")
    log.info(f"   Bisiklet ile:         {result['co2_utilise_g']} g CO₂")
    log.info(f"   TASARRUF:             {result['co2_economise_g']} g CO₂ = {result['co2_economise_kg']} kg")
    log.info(f"   Ağaç eşdeğeri:        {result['arbres_equivalent']} ağaç/yıl")

    # Avant/Après karşılaştırması
    log.info("")
    log.info("   📊 AVANT / APRÈS Modality-Flow:")
    log.info("   ┌─────────────────────────────────────────┐")
    log.info("   │ AVANT  → Herkes araba kullanıyor        │")
    log.info(f"   │         {result['co2_reference_g']:>8.0f} g CO₂/ay/kullanıcı  │")
    log.info("   │ APRÈS  → Modality-Flow ile bisiklet     │")
    log.info(f"   │         {result['co2_utilise_g']:>8.0f} g CO₂/ay/kullanıcı  │")
    log.info(f"   │ GAIN   → {result['co2_economise_g']:>8.0f} g CO₂/ay/kullanıcı  │")
    log.info("   └─────────────────────────────────────────┘")

    # Ölçekli projeksiyon
    n_users = 7500  # Montpellier %5 penetrasyon
    co2_total_kg = result["co2_economise_kg"] * n_users * 12
    log.info(f"\n   Projeksiyon (7.500 kullanıcı, 1 yıl):")
    log.info(f"   → {co2_total_kg:,.0f} kg CO₂ tasarruf = {co2_total_kg/1000:,.1f} tonnes")


# ══════════════════════════════════════════════════════════════════
# 5. AVANT/APRÈS ANALİZİ
# ══════════════════════════════════════════════════════════════════

def avant_apres_analysis():
    """
    Hava kalitesi ve trafik üzerinde Modality-Flow etkisi.
    Gerçek AQI verisini kullan + projeksiyon yap.
    """
    log.info("=" * 65)
    log.info("AVANT / APRÈS — Impact Modality-Flow")
    log.info("=" * 65)

    con = duckdb.connect(str(DUCKDB_PATH))

    # AQI istatistikleri
    df_aqi = con.execute("""
        SELECT
            EXTRACT(MONTH FROM date) AS mois,
            AVG(indice_qualite) AS aqi_moyen,
            MIN(indice_qualite) AS aqi_min,
            MAX(indice_qualite) AS aqi_max,
            COUNT(*) AS nb_jours
        FROM dim_qualite_air
        GROUP BY mois
        ORDER BY mois
    """).fetchdf()

    log.info(f"\n   Qualité de l'air Montpellier (2024):")
    log.info(f"{df_aqi.to_string(index=False)}")

    # Bisiklet kullanım analizi
    df_usage = con.execute("""
        SELECT
            EXTRACT(HOUR FROM timestamp) AS heure,
            AVG(bisiklet_sayisi) AS moy_bisiklet,
            COUNT(*) AS nb_mesures
        FROM fact_velomagg_historique
        GROUP BY heure
        ORDER BY heure
    """).fetchdf()

    log.info(f"\n   Utilisation moyenne Vélomagg par heure:")
    log.info(f"{df_usage.to_string(index=False)}")

    # Heure de pointe analizi
    df_pointe = con.execute("""
        SELECT
            CASE
                WHEN EXTRACT(HOUR FROM timestamp) BETWEEN 7 AND 9  THEN 'Matin (7-9h)'
                WHEN EXTRACT(HOUR FROM timestamp) BETWEEN 12 AND 14 THEN 'Midi (12-14h)'
                WHEN EXTRACT(HOUR FROM timestamp) BETWEEN 17 AND 19 THEN 'Soir (17-19h)'
                ELSE 'Hors pointe'
            END AS periode,
            AVG(bisiklet_sayisi) AS moy_bisiklet,
            COUNT(DISTINCT station_id) AS nb_stations
        FROM fact_velomagg_historique
        GROUP BY periode
        ORDER BY moy_bisiklet
    """).fetchdf()

    log.info(f"\n   Utilisation par période:")
    log.info(f"{df_pointe.to_string(index=False)}")

    con.close()


# ══════════════════════════════════════════════════════════════════
# RAPPORT FINAL
# ══════════════════════════════════════════════════════════════════

def save_report(model, le, mae, r2):
    """ML raporu kaydet."""
    report = {
        "timestamp": datetime.now().isoformat(),
        "model_1_availability": {
            "type": "RandomForestRegressor",
            "mae": round(mae, 2),
            "r2": round(r2, 3),
            "features": le.classes_.tolist() if hasattr(le, 'classes_') else []
        },
        "model_2_route": {
            "type": "Rule-based CO₂ optimization",
            "co2_factors": CO2_FACTORS
        },
        "model_3_savings": {
            "type": "CO₂ savings calculator",
            "reference": "ADEME 2024"
        }
    }

    import json
    path = REPORTS_DIR / "ml_report.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    log.info(f"Rapport sauvegardé: {path}")


# ══════════════════════════════════════════════════════════════════
# POINT D'ENTRÉE
# ══════════════════════════════════════════════════════════════════

def run():
    start = datetime.now()
    log.info("MODALITY-FLOW ML Pipeline démarré")
    log.info(f"   Heure: {start.strftime('%Y-%m-%d %H:%M:%S')}")

    # 1. Veri yükle
    df = load_features()

    # 2. Model 1 — Bisiklet doluluk tahmini
    model, le = train_availability_model(df)

    # MAE ve R² skorları için tekrar hesapla
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error, r2_score
    FEATURES = ["heure", "jour_semaine", "mois", "jour_mois",
                "heure_pointe", "weekend", "temperature_max",
                "precipitation_sum", "wind_speed_max",
                "indice_qualite", "no2", "o3", "pm10", "station_encoded"]
    X = df[FEATURES]
    y = df["bisiklet_sayisi"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    # 3. Model 2 — Rota optimizasyonu
    demo_route_optimization()

    # 4. Model 3 — CO₂ tasarrufu
    demo_co2_savings()

    # 5. Avant/Après analizi
    avant_apres_analysis()

    # 6. Rapor
    save_report(model, le, mae, r2)

    elapsed = (datetime.now() - start).seconds
    log.info("=" * 65)
    log.info(f"ML Pipeline terminé en {elapsed}s")
    log.info(f"   Modèles: {MODELS_DIR}")
    log.info(f"   Rapports: {REPORTS_DIR}")
    log.info("=" * 65)


if __name__ == "__main__":
    run() 

2026-05-13 10:56:39 [INFO] MODALITY-FLOW ML Pipeline démarré
2026-05-13 10:56:39 [INFO]    Heure: 2026-05-13 10:56:39
2026-05-13 10:56:39 [INFO] =================================================================
2026-05-13 10:56:39 [INFO] Veri yükleniyor...
2026-05-13 10:56:39 [INFO] =================================================================
2026-05-13 10:56:39 [INFO] 644880 satır yüklendi
2026-05-13 10:56:39 [INFO]    Kolonlar: ['station_id', 'timestamp', 'bisiklet_sayisi', 'heure', 'jour_semaine', 'mois', 'jour_mois', 'heure_pointe', 'weekend', 'temperature_max', 'temperature_min', 'precipitation_sum', 'wind_speed_max', 'weather_code', 'indice_qualite', 'no2', 'o3', 'pm10']
2026-05-13 10:56:39 [INFO]    İstasyon sayısı: 57
2026-05-13 10:56:39 [INFO]    Tarih aralığı: 2024-01-01 01:00:13 → 2026-05-13 01:59:13
2026-05-13 10:56:39 [INFO]    Bisiklet min/max: 0 / 55
2026-05-13 10:56:39 [INFO] =================================================================
2026-05-13 10:56:39 [INF